<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/HW/JoshProfeta/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
def build_system(V0, R1, R2, R3, R4, R5):
  """ Solve a bridge circuit with five resistors and one voltage source.

  The circuit has the topography:
      B ---- R4 ---- C--(-)
      | L            |
      |  L           |
      R3  L__R5 _    R2
      |           L  |
      |            L |
  (+)-A ---- R1 ---- D

  Uses Kirchhoff's Current and Voltage Laws to create
  a system of linear equations A*I = b.

  Parameters
  ----------
  V0: float
    Voltage of source between A and C (V)
  R1, R2, R3, R4: float
    Resistances of the bridge resistors (Ohms)
  R5: float
    Galvanometer (bridge resistor between B and D) resistance (Ohms)

  Returns
  -------
  Currents : numpy.ndarray
    Array [I1 I2 I3 I4 I5] of branch currents (A).
  """
  # Assume current directions:
  # I1: V0 -> R1 -> Node D
  # I2: Node D -> R2 -> Node C (Ground)
  # I3: V0 -> R3 -> Node B
  # I4: Node B -> R4 -> Node C (Ground)
  # I5: Node D -> R5 -> Node B

  # Formulate equations based on KCL at nodes B and D, and KVL for three separate loops.
  # Equations:
  # 1. KCL at Node B: I3 - I4 + I5 = 0
  # 2. KCL at Node D: I1 - I2 - I5 = 0
  # 3. KVL Loop V0-R1-R2-C-V0: V0 - I1*R1 - I2*R2 = 0 => V0 = I1*R1 + I2*R2
  # 4. KVL Loop V0-R3-R4-C-V0: V0 - I3*R3 - I4*R4 = 0 +> V0 = I3*R3 + I4*R4
  # 5. KVL Loop D-R5-B-R4-R2-D: I5*R5 + I4*R4 - I2*R2 = 0

  import numpy as np

  A = np.array([
      [1, -1, 0, 0, -1],  # KCL at Node D (I1 - I2 - I5 = 0)
      [0, 0, 1, -1, 1],   # KCL at Node B (I3 - I4 + I5 = 0)
      [R1, R2, 0, 0, 0],  # KVL Loop A-D-C-A (R1*I1 + R2*I2 = V0)
      [0, 0, R3, R4, 0],  # KVL Loop A-B-C-A (R3*I3 + R4*I4 = V0)
      [0, -R2, 0, R4, R5] # KVL Loop B-C-D-B (-R2*I2 + R4*I4 + R5*I5 = 0)
  ], dtype=float)

  b = np.array([0, 0, V0, V0, 0], dtype=float)

  print("Matrix A:")
  print(A)
  print("Vector b:")
  print(b)

# Given Values
V0 = 10 # Volts
R1 = 100 # Ohms
R2 = 200 # Ohms
R3 = 300 # Ohms
R4 = 600 # Ohms
R5 = 50 # Ohms

build_system(V0, R1, R2, R3, R4, R5)

def solve_circuit(A, b):

  import numpy as np
  currents = np.linalg.solve(A, b)

# Given Values
V0 = 10 # Volts
R1 = 100 # Ohms
R2 = 200 # Ohms
R3 = 300 # Ohms
R4 = 600 # Ohms
R5 = 50 # Ohms

import numpy as np
A = np.array([
      [1, -1, 0, 0, -1],
      [0, 0, 1, -1, 1],
      [R1, R2, 0, 0, 0],
      [0, 0, R3, R4, 0],
      [0, -R2, 0, R4, R5]
  ], dtype=float)

b = np.array([0, 0, V0, V0, 0], dtype=float)

solve_circuit(A, b)

print("Branch Currents:")
print(f"I1 = {currents[0]:.4f} A (through R1)")
print(f"I2 = {currents[1]:.4f} A (through R2)")
print(f"I3 = {currents[2]:.4f} A (through R3)")
print(f"I4 = {currents[3]:.4f} A (through R4)")
print(f"I5 = {currents[4]:.4f} A (through R5)")

Matrix A:
[[   1.   -1.    0.    0.   -1.]
 [   0.    0.    1.   -1.    1.]
 [ 100.  200.    0.    0.    0.]
 [   0.    0.  300.  600.    0.]
 [   0. -200.    0.  600.   50.]]
Vector b:
[ 0.  0. 10. 10.  0.]
Branch Currents:
I1 = 0.0333 A (through R1)
I2 = 0.0333 A (through R2)
I3 = 0.0111 A (through R3)
I4 = 0.0111 A (through R4)
I5 = -0.0000 A (through R5)


In [33]:
from enum import verify
def verify_power_balance(V0, R1, R2, R3, R4, R5, currents):
  """ Verify conservation of energy in the circuit.

  Tests to ensure that the power supplied by the voltage source equals the total power dissipated in all resistors.

  Parameters
  ----------
  V0: float
      Voltage of the source (V)
  R1, R2, R3, R4, R5: float
      Resistances of the resistors (Ohms)
  currents : numpy.ndarray
      Array [I1 I2 I3 I4 I5] of branch currents (A).

  Returns
  -------
  bool
      True if power balance is satisfied (within numerical tolerance)
  """
  I1, I2, I3, I4, I5 = currents
  P_source = V0 * (I1 + I3)
  P_dissipated = I1**2*R1 + I2**2*R2 + I3**2*R3 + I4**2*R4 + I5**2*R5

  print(f"Power delivered by source:      {P_source:.4f} W")
  print(f"Power dissipated in resistors:  {P_dissipated:.4f} W")

  return np.isclose(P_source, P_dissipated)

balanced = verify_power_balance(V0, R1, R2, R3, R4, R5, currents)
print(f"Energy conserved: {balanced}")

Power delivered by source:      0.4444 W
Power dissipated in resistors:  0.4444 W
Energy conserved: True
